# 2025 Tariff Revenue EstimateEstimates tariff revenue using 2025 import values from Census.Computes interval-by-interval revenue, split by authority (IEEPA vs. Section 232).Annual imports are spread uniformly across the year; tariff rates from the policy engine.**Data pipeline**: This notebook loads its own copy of the tariff engine(country list, HS-code lists, masks, apply functions, action registry)so it can compute product-level tariff rates for each policy interval.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import pyarrow as pa
import pyarrow.parquet as pq
import os
import warnings
warnings.filterwarnings('ignore')

## Data Pipeline### Country list & baseline rates

In [ ]:
country_list = pd.read_csv('./data/top_50_non_eu.csv', dtype={'CTY_CODE': str})

# Add EU as a bloc
eu_row = pd.DataFrame([{"CTY_NAME": "EUROPEAN UNION", "CTY_CODE": "0003"}])
country_list = pd.concat([country_list, eu_row], ignore_index=True)

# Initial baseline tariff (used for Apr 5 reciprocal activation)
country_list["tariff"] = 10.0
country_list.loc[country_list["CTY_CODE"] == "5700", "tariff"] = 30.0   # China
country_list.loc[country_list["CTY_CODE"] == "5820", "tariff"] = 30.0   # Hong Kong
country_list.loc[country_list["CTY_CODE"] == "1220", "tariff"] = 8.75   # Canada (USMCA-adjusted)
country_list.loc[country_list["CTY_CODE"] == "2010", "tariff"] = 4.5    # Mexico (USMCA-adjusted)
country_list.loc[country_list["CTY_CODE"] == "0003", "tariff"] = 10.0   # EU

# Merge Annex I reciprocal rates
reciprocal = pd.read_csv('./tariff-lists/reciprocal-tariffs-annex-I.csv', dtype={'CTY_CODE': str})
reciprocal.rename(columns={"tariff_rate": "rtariff"}, inplace=True)
country_list = pd.merge(
    left=country_list, right=reciprocal[["rtariff", "CTY_CODE"]],
    on="CTY_CODE", how="left"
)
country_list["rtariff"] = country_list["rtariff"].fillna(10.0)
country_list.loc[country_list["CTY_CODE"] == "5700", "rtariff"] = 34.0  # China
country_list.loc[country_list["CTY_CODE"] == "5820", "rtariff"] = 34.0  # Hong Kong

# Merge August 7 updated rates (from EO 14326 / bilateral letters)
new_list = pd.read_csv('./tariff-lists/clean-august-7-tariffs.csv')
new_list.rename({"country_name": "CTY_NAME"}, axis=1, inplace=True)
country_list = pd.merge(country_list, new_list, on="CTY_NAME", how="left")
country_list["rtariff-aug"] = country_list["rtariff-aug"].fillna(country_list["rtariff"])
country_list.loc[country_list["CTY_NAME"] == "KOREA, SOUTH", "rtariff-aug"] = 15.0
country_list.loc[country_list["CTY_NAME"] == "EUROPEAN UNION", "rtariff-aug"] = 15.0
country_list.loc[country_list["CTY_NAME"] == "INDIA", "rtariff-aug"] = 50.0

### Product-level tariff / exemption lists

In [ ]:
exemption_list = pd.read_csv(
    './tariff-lists/annex-II-exemptions.csv', dtype={'HTSUS': str})
# Annex II of EO 14257 — products exempt from reciprocal tariffs

exemption_list_phones = pd.read_csv(
    './tariff-lists/consumer_electronics_EO14257_apr11.csv', dtype={'HTSUS': str})
# Apr 11 consumer-electronics exemption

steel_list = pd.read_csv(
    './tariff-lists/steel-tariffs.csv', dtype={'HTSUS': str})
# Section 232 steel products (HS8) — Proclamation 10896

alu_list = pd.read_csv(
    './tariff-lists/alu-tariffs.csv', dtype={'HTSUS': str})
# Section 232 aluminum products (mixed HS8/HS10) — Proclamation 10895
alu_8_list = alu_list[alu_list['HTSUS'].str.len() == 8]

auto_list = pd.read_csv(
    './tariff-lists/auto-tariffs.csv', dtype={'HTSUS': str})
# Section 232 auto tariffs (mixed HS6/HS8/HS10) — Proclamation 10908
auto_6_list = auto_list[auto_list['HTSUS'].str.len() == 6]
auto_8_list = auto_list[auto_list['HTSUS'].str.len() == 8]

copper_list = pd.read_csv(
    './tariff-lists/copper-list.csv', dtype={'HTSUS': str})
# Section 232 copper products (HS8) — Proclamation 10962

brazil_nonaircraft_exemptions = pd.read_csv(
    './tariff-lists/brazil-non-airplane-exemptions.csv', dtype={'HTSUS': str})
brazil_aircraft_exemptions = pd.read_csv(
    './tariff-lists/brazil-airplane-exemptions.csv', dtype={'HTSUS': str})
# Brazil-specific exemptions from EO 14323

nov_exemptions_list = pd.read_csv(
    './tariff-lists/annex-II-exemptions-nov14.csv', dtype={'HTSUS': str})
# Nov 14 agricultural exemptions (EO 14360)

s122_exempt_list = pd.read_csv(
    './tariff-lists/section122-product-exemptions.csv', dtype={'hts_code': str})
# Section 122 product exemptions, Annex I subdivision (aa)(ii) — 1,098 HS codes
# Covers energy (Ch 27), critical minerals, pharma/chem, ag, electronics,
# fertilizers, bullion, natural rubber, base metals, semiconductors, etc.
s122_exempt_8 = s122_exempt_list[s122_exempt_list['digits'] == 8]['hts_code'].tolist()
s122_exempt_10 = s122_exempt_list[s122_exempt_list['digits'] == 10]['hts_code'].tolist()


# ── April 2026 Section 232 Metals Restructuring (Proclamation, Apr 2, 2026) ──
metals_ia_list = pd.read_csv(
    './tariff-lists/metals-annex-IA.csv', dtype={'HTSUS': str})
# Annex I-A: 50% tier — primary metals + certain derivatives
metals_ia_4  = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 4]['HTSUS'].tolist()
metals_ia_6  = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 6]['HTSUS'].tolist()
metals_ia_8  = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 8]['HTSUS'].tolist()
metals_ia_10 = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 10]['HTSUS'].tolist()

metals_ib_list = pd.read_csv(
    './tariff-lists/metals-annex-IB.csv', dtype={'HTSUS': str})
# Annex I-B: 25% tier — copper articles + steel/alu derivative products
metals_ib_8  = metals_ib_list[metals_ib_list['HTSUS'].str.len() == 8]['HTSUS'].tolist()
metals_ib_10 = metals_ib_list[metals_ib_list['HTSUS'].str.len() == 10]['HTSUS'].tolist()

metals_ii_list = pd.read_csv(
    './tariff-lists/metals-annex-II-removed.csv', dtype={'HTSUS': str})
# Annex II: removed from scope of Section 232
metals_ii_8  = metals_ii_list[metals_ii_list['HTSUS'].str.len() == 8]['HTSUS'].tolist()
metals_ii_10 = metals_ii_list[metals_ii_list['HTSUS'].str.len() == 10]['HTSUS'].tolist()

metals_iii_list = pd.read_csv(
    './tariff-lists/metals-annex-III-temporary.csv', dtype={'HTSUS': str})
# Annex III: temporary ~15% rate through Dec 31, 2027
metals_iii_8  = metals_iii_list[metals_iii_list['HTSUS'].str.len() == 8]['HTSUS'].tolist()
metals_iii_10 = metals_iii_list[metals_iii_list['HTSUS'].str.len() == 10]['HTSUS'].tolist()


### Import data functions

In [ ]:
# ── Import Data: Download and Load Functions ─────────────────────────────

CENSUS_API_KEY = "34e40301bda77077e24c859c6c6c0b721ad73fc7"

def download_imports(cnty_code, year=2024, api_key=CENSUS_API_KEY):
    """
    Pull HS10-level import data from the Census API for a given country
    and December of the specified year. Computes import shares (weights).
    
    Parameters
    ----------
    cnty_code : str — Census country code
    year : int — year for import weights (e.g. 2024, 2025)
    api_key : str — Census API key
    
    Returns
    -------
    df : DataFrame with columns including HS10, HS8, HS6, imports, duty, share
    """
    import requests
    
    time_period = f"{year}-12"
    
    url = (
        f"https://api.census.gov/data/timeseries/intltrade/imports/hs"
        f"?get=CTY_NAME,CON_VAL_YR,CAL_DUT_YR,I_COMMODITY,I_COMMODITY_SDESC"
        f"&key={api_key}"
        f"&time={time_period}"
        f"&COMM_LVL=HS10"
        f"&CTY_CODE={cnty_code}"
    )
    
    r = requests.get(url)
    r.raise_for_status()
    
    df = pd.DataFrame(r.json()[1:], columns=r.json()[0])
    
    df["time"] = pd.to_datetime(df["time"], format="%Y-%m")
    df["imports"] = df["CON_VAL_YR"].astype(float)
    df["duty"] = df["CAL_DUT_YR"].astype(float)
    df["share"] = df["imports"] / df["imports"].sum()
    df["HS10"] = df["I_COMMODITY"].astype(str)
    df["HS8"] = df["HS10"].str[0:8]
    df["HS6"] = df["HS10"].str[0:6]
    
    return df


def download_all_imports(country_list, year=2024, api_key=CENSUS_API_KEY, data_dir="./data"):
    """
    Download HS10 import data for all countries and cache as parquet files.
    File naming convention: {data_dir}/{year}-hs10-imports{CTY_CODE}.parquet
    
    Parameters
    ----------
    country_list : DataFrame — must have 'CTY_CODE' column
    year : int — year for import weights
    api_key : str — Census API key
    data_dir : str — directory for parquet files
    """
    import pyarrow as pa
    import pyarrow.parquet as pq
    
    for _, row in country_list.iterrows():
        print(f"Downloading {row.get('CTY_NAME', row['CTY_CODE'])}...")
        df = download_imports(row['CTY_CODE'], year=year, api_key=api_key)
        outfile = f"{data_dir}/{year}-hs10-imports{row['CTY_CODE']}.parquet"
        pq.write_table(pa.Table.from_pandas(df), outfile)
    
    print("Done.")


def load_imports(cnty_code, year=2024, data_dir="./data"):
    """
    Load cached HS10-level import data from parquet.
    
    Parameters
    ----------
    cnty_code : str — Census country code
    year : int — weight year (must match what was downloaded)
    data_dir : str — directory for parquet files
    
    Returns
    -------
    df : DataFrame
    """
    infile = f"{data_dir}/{year}-hs10-imports{cnty_code}.parquet"
    return pq.read_table(infile).to_pandas()

### Product-group masks

In [ ]:
# ── Product-group masks ──────────────────────────────────────────────────

def _mask_steel(df):
    return df["HS8"].isin(steel_list["HTSUS"].tolist())

def _mask_alu(df):
    return (df["HS10"].isin(alu_list["HTSUS"].tolist())
            | df["HS8"].isin(alu_8_list["HTSUS"].tolist()))

def _mask_auto(df):
    return (df["HS8"].isin(auto_8_list["HTSUS"].tolist())
            | df["HS10"].isin(auto_list["HTSUS"].tolist())
            | df["HS6"].isin(auto_6_list["HTSUS"].tolist()))

def _mask_copper(df):
    return df["HS8"].isin(copper_list["HTSUS"].tolist())

def _mask_exemptions(df):
    return df["HS10"].isin(exemption_list["HTSUS"].tolist())

def _mask_phones(df):
    return df["HS10"].isin(exemption_list_phones["HTSUS"].tolist())

def _mask_brazil_non_aircraft(df):
    return df["HS8"].isin(brazil_nonaircraft_exemptions["HTSUS"].tolist())

def _mask_brazil_aircraft(df):
    return df["HS8"].isin(brazil_aircraft_exemptions["HTSUS"].tolist())

def _mask_nov_exemptions(df):
    return df["HS8"].isin(nov_exemptions_list["HTSUS"].tolist())

def _mask_s122_exempt(df):
    """Products exempt from Section 122 surcharge under Annex I (aa)(ii).
    Mixed HS8 (1,097 codes) and HS10 (1 code: 8505110070 NdFeB magnets).
    Aircraft exemptions (aa)(iv) are ignored — see design notes."""
    return df["HS8"].isin(s122_exempt_8) | df["HS10"].isin(s122_exempt_10)


# ── April 2026 Metals Restructuring Masks ────────────────────────────────

def _mask_annex_IA(df):
    """Annex I-A: 50% tier. Mixed HS4/HS6/HS8/HS10 codes."""
    return (df["HS10"].isin(metals_ia_10)
            | df["HS8"].isin(metals_ia_8)
            | df["HS6"].isin(metals_ia_6)
            | df["HS8"].str[:4].isin(metals_ia_4))

def _mask_annex_IB(df):
    """Annex I-B: 25% tier. HS8/HS10 codes."""
    return (df["HS10"].isin(metals_ib_10)
            | df["HS8"].isin(metals_ib_8))

def _mask_annex_II_removed(df):
    """Annex II: removed from scope. HS8/HS10 codes."""
    return (df["HS10"].isin(metals_ii_10)
            | df["HS8"].isin(metals_ii_8))

def _mask_annex_III(df):
    """Annex III: temporary ~15% rate. HS8/HS10 codes."""
    return (df["HS10"].isin(metals_iii_10)
            | df["HS8"].isin(metals_iii_8))

def _is_alu_chapter(df):
    """Helper: True for products in HS Chapter 76 (aluminum).
    Used to identify Russian aluminum for the 200% rate (Clause 8)."""
    return df["HS8"].str[:2] == "76"


### Tariff apply functions & action registry

In [ ]:
# ── Phase 1: Base country-level rate functions ──────────────────────────
# These set the IEEPA layer (blanket country rates).
# Phase 2 sector overrides will stamp on top of these.

def _apply_fentanyl_feb(df, cty, rate):
    """Feb 4: IEEPA fentanyl tariffs — 20% on China/HK."""
    df["tariff"] = 0.0
    if cty in ("5700", "5820"):
        df["tariff"] = 20.0
    return df


def _apply_usmca_march(df, cty, rate):
    """Mar 6: USMCA carve-outs. Canada 8.75%, Mexico 4.5%; China/HK 20%."""
    df["tariff"] = 0.0
    if cty in ("5700", "5820"):
        df["tariff"] = 20.0
    elif cty == "1220":
        df["tariff"] = 8.75
    elif cty == "2010":
        df["tariff"] = 4.5
    return df


def _apply_reciprocal_apr5(df, cty, rate):
    """Apr 5: Reciprocal tariffs activate at country-specific rates."""
    df["tariff"] = rate
    if cty in ("5700", "5820"):
        df["tariff"] = rate + 20.0  # fentanyl stacks
    elif cty == "1220":
        df["tariff"] = 8.75
    elif cty == "2010":
        df["tariff"] = 4.5
    return df


def _apply_china_escalation(df, cty, rate):
    """Apr 8: China escalated to 84% reciprocal (+20% fentanyl = 104%)."""
    df["tariff"] = rate
    if cty in ("5700", "5820"):
        df["tariff"] = 84.0 + 20.0
    elif cty == "1220":
        df["tariff"] = 8.75
    elif cty == "2010":
        df["tariff"] = 4.5
    return df


def _apply_90day_pause(df, cty, rate):
    """Apr 9: 90-day pause. Universal 10%; China/HK at 125%."""
    df["tariff"] = 10.0
    if cty in ("5700", "5820"):
        df["tariff"] = 125.0
    elif cty == "1220":
        df["tariff"] = 8.75
    elif cty == "2010":
        df["tariff"] = 4.5
    return df


def _apply_geneva_deal(df, cty, rate):
    """May 12: Geneva deal — China from 125% to 30% (10% recip + 20% fentanyl)."""
    df["tariff"] = 10.0
    if cty == "1220":
        df["tariff"] = 8.75
    elif cty == "2010":
        df["tariff"] = 4.5
    elif cty in ("5700", "5820"):
        df["tariff"] = 30.0
    return df


def _apply_august_base_rates(df, cty, rate):
    """Aug 7: Country-specific reciprocal rates from EO 14326.
    Canada fentanyl increased to 35% (EO 14325).
    Brazil +40% (EO 14323)."""
    df["tariff"] = rate
    if cty == "1220":  # Canada
        df["tariff"] = (8.75 / 25.0) * 35.0
    if cty == "3510":  # Brazil
        df["tariff"] = 50.0
    return df


def _apply_november_china_base(df, cty, rate):
    """Nov 1: China fentanyl reduced 20% -> 10%.
    Net China rate: 10% reciprocal + 10% fentanyl = 20%.
    Canada fentanyl rate scaled up."""
    if cty in ("5700", "5820"):
        df["tariff"] = 20.0  # hardcoded: 10% reciprocal + 10% fentanyl
    if cty == "1220":
        df["tariff"] = (8.75 / 25.0) * 45.0
    return df


def _apply_november_swiss_base(df, cty, rate):
    """Nov 14: Switzerland rate capped at 15%."""
    if cty == "4419":
        df["tariff"] = 15.0
    return df


def _apply_korea_base(df, cty, rate):
    """Dec 4: South Korea deal — 15% reciprocal ceiling."""
    if cty == "5800":
        df["tariff"] = 15.0
    return df


def _apply_india_deal(df, cty, rate):
    """Feb 7, 2026: India — Russian oil tariff removed, reciprocal to 18%."""
    if cty == "5330":
        df["tariff"] = 18.0
    return df


def _apply_scotus_base(df, cty, rate):
    """Feb 20, 2026: EO revoking all IEEPA ad valorem duties.
    Revokes EO 14193/14194/14195 (fentanyl), EO 14257 (reciprocal),
    EO 14323 (Brazil), and all bilateral amendments.
    Section 232 and Section 301 duties are explicitly preserved."""
    df["tariff"] = 0.0
    return df


# ── Phase 2: Sector overrides and exemptions ────────────────────────────
# These stamp on top of the phase-1 base rates. Section 232 tariffs and
# product exemptions are applied here so they are never wiped by IEEPA
# base-rate resets.

def _apply_steel_alu_25(df, cty, rate):
    """Mar 12: Section 232 steel and aluminum at 25%."""
    df.loc[_mask_steel(df), "tariff"] = 25.0
    df.loc[_mask_alu(df), "tariff"] = 25.0
    return df


def _apply_auto_initial(df, cty, rate):
    """Apr 3: Section 232 autos at 25% (Canada/Mexico exempt)."""
    
    if cty in ("1220", "2010"):       # Canada/Mexico — seem lower
        df.loc[_mask_auto(df), "tariff"] = 15.0
    else:
        df.loc[_mask_auto(df), "tariff"] = 25.0

    return df
    # if cty not in ("1220", "2010"): # previous version: all non-Canada/Mexico autos at 25%
    #     df.loc[_mask_auto(df), "tariff"] = 25.0
    # return df


def _apply_phone_exemption(df, cty, rate):
    """Apr 11: Consumer electronics exempt (China still pays fentanyl 20%)."""
    phones = _mask_phones(df)
    if cty == "5700":
        df.loc[phones, "tariff"] = 20.0
    else:
        df.loc[phones, "tariff"] = 0.0
    return df


def _apply_annex_ii(df, cty, rate):
    """Annex II exemptions from reciprocal tariffs."""
    df.loc[_mask_exemptions(df), "tariff"] = 0.0
    return df


def _apply_steel_alu_50(df, cty, rate):
    """Jun 3: Steel/aluminum doubled to 50%. UK stays at 25% per deal."""
    if cty == "4120":  # UK
        df.loc[_mask_steel(df), "tariff"] = 25.0
        df.loc[_mask_alu(df), "tariff"] = 25.0
    else:
        df.loc[_mask_steel(df), "tariff"] = 50.0
        df.loc[_mask_alu(df), "tariff"] = 50.0
    return df


def _apply_august_sectors(df, cty, rate):
    """Aug 7: Copper 50% (Proc 10962); auto deals; Brazil exemptions."""
    # Section 232: Copper at 50%
    df.loc[_mask_copper(df), "tariff"] = 50.0

    # Section 232: Auto deals (bilateral adjustments to Proc 10908)
    auto = _mask_auto(df)
    if cty in ("1220", "2010"):       # Canada/Mexico — seem to be at 15%
        df.loc[auto, "tariff"] = 15.0
    elif cty == "5880":               # Japan — 15%
        df.loc[auto, "tariff"] = 15.0
    elif cty == "0003":               # EU — 15%
        df.loc[auto, "tariff"] = 15.0
    elif cty == "5800":               # South Korea — 15%
        df.loc[auto, "tariff"] = 15.0
    else:
        df.loc[auto, "tariff"] = 25.0

    # Brazil exemptions (EO 14323 Annex)
    if cty == "3510":
        df.loc[_mask_brazil_non_aircraft(df), "tariff"] = 0.0
        df.loc[_mask_brazil_aircraft(df), "tariff"] = 0.0

    return df


def _apply_november_ag_phones(df, cty, rate):
    """Nov 14: Agricultural exemptions (EO 14360); China phone fentanyl adjustment."""
    df.loc[_mask_nov_exemptions(df), "tariff"] = 0.0
    if cty == "5700":
        df.loc[_mask_phones(df), "tariff"] = 10.0  # fentanyl reduced to 10%
    return df


def _apply_korea_auto(df, cty, rate):
    """Dec 4: South Korea auto deal at 15%."""
    if cty == "5800":
        df.loc[_mask_auto(df), "tariff"] = 15.0
    return df


def _apply_scotus_sectors(df, cty, rate):
    """Feb 20, 2026: Re-apply only surviving Section 232 tariffs.
    All IEEPA tariffs revoked by EO; 232 explicitly preserved."""
    # Steel 50% (UK 25%) — Proc 10896/10947
    steel = _mask_steel(df)
    if cty == "4120":
        df.loc[steel, "tariff"] = 25.0
    else:
        df.loc[steel, "tariff"] = 50.0

    # Aluminum 50% (UK 25%) — Proc 10895/10947
    alu = _mask_alu(df)
    if cty == "4120":
        df.loc[alu, "tariff"] = 25.0
    else:
        df.loc[alu, "tariff"] = 50.0

    # Autos with deal rates — Proc 10908 as amended
    auto = _mask_auto(df)
    if cty in ("1220", "2010"):
        df.loc[auto, "tariff"] = 15.0 # seem to be about 15%
    elif cty == "4120":
        df.loc[auto, "tariff"] = 10.0
    elif cty in ("5880", "0003", "5800"):
        df.loc[auto, "tariff"] = 15.0
    else:
        df.loc[auto, "tariff"] = 25.0

    # Copper 50% — Proc 10962
    df.loc[_mask_copper(df), "tariff"] = 50.0

    # IEEPA phone exemption and fentanyl surcharge on phones both struck down —
    # zero out any phone rate set by earlier phase-2 actions
    df.loc[_mask_phones(df), "tariff"] = 0.0

    return df



# ── Section 122 Temporary Import Surcharge (Feb 24, 2026) ───────────────

def _apply_s122_base(df, cty, rate):
    """Feb 24, 2026: Section 122 temporary import surcharge — 10% universal.
    Authority: Section 122, Trade Act of 1974 (balance-of-payments emergency).
    Duration: 150 days (through Jul 24, 2026) unless extended by Congress.
    Canada/Mexico USMCA-compliant goods are exempt per (aa)(vi)/(vii);
    we treat all Canada/Mexico imports as USMCA-eligible (same simplification
    used throughout)."""
    if cty in ("1220", "2010"):  # Canada, Mexico — USMCA exempt
        pass  # tariff stays at 0 from ieepa_revocation_base
    else:
        df["tariff"] = 10.0 
        #df["tariff"] = 15.0 # I went to the Edina Girls 3rd place State Hockey Game
                            # came back and found out I had to change it to 15%
    return df


def _apply_s122_exemptions(df, cty, rate):
    """Feb 24, 2026: Zero out Section 122 product exemptions from Annex I (aa)(ii).
    These 1,098 HS codes cover energy, critical minerals, pharma, ag, electronics,
    fertilizers, bullion, semiconductors, etc.
    Section 232 products are explicitly carved out of the surcharge per (aa)(v) —
    they keep their 232 rates and pay 0% surcharge on covered content."""
    if cty in ("1220", "2010"):
        return df  # Already fully exempt via USMCA
    exempt = _mask_s122_exempt(df)
    is_232 = (_mask_steel(df) | _mask_alu(df) | _mask_auto(df) | _mask_copper(df)
             | _mask_annex_IA(df) | _mask_annex_IB(df) | _mask_annex_III(df))
    df.loc[exempt & ~is_232, "tariff"] = 0.0
    return df



# ── April 2026 Section 232 Metals Tiered Restructuring ──────────────────

def _apply_metals_restructuring(df, cty, rate):
    """Apr 6, 2026: Section 232 metals tiered restructuring.
    Replaces flat 50% with tiered system per Proclamation of Apr 2, 2026.
    
    Clause (2): Annex I-A → 50% (UK 25%)
    Clause (3): Annex I-B → 25% (UK 15%)
    Clause (5): Annex III → ~15% flat approx (temporary through Dec 31, 2027)
    Clause (4): Annex II → 0% (removed from scope)
    Clause (8): Russian aluminum → 200% (Proc 10522, unchanged)
    Clause (9): Products on multiple lists pay once, not combined
    Clause (10): Prior bilateral auto agreements preserved — skip autos
    """
    # Identify auto products to skip (Clause 10: does not alter prior agreements)
    auto = _mask_auto(df)

    # --- Annex I-A: 50% tier (Clause 2) ---
    ia = _mask_annex_IA(df) & ~auto
    if cty == "4120":       # UK: 25% per Clause (2)(b)
        df.loc[ia, "tariff"] = 25.0
    elif cty == "4621":     # Russia
        # Clause (8): ALL Russian aluminum at 200% (Proc 10522)
        df.loc[ia & _is_alu_chapter(df), "tariff"] = 200.0
        df.loc[ia & ~_is_alu_chapter(df), "tariff"] = 50.0
    else:
        df.loc[ia, "tariff"] = 50.0

    # --- Annex I-B: 25% tier (Clause 3) ---
    ib = _mask_annex_IB(df) & ~auto & ~ia  # Clause (9): single application
    if cty == "4120":       # UK: 15% per Clause (3)(b)
        df.loc[ib, "tariff"] = 15.0
    elif cty == "4621":     # Russia
        df.loc[ib & _is_alu_chapter(df), "tariff"] = 200.0  # Clause (8)
        df.loc[ib & ~_is_alu_chapter(df), "tariff"] = 25.0
    else:
        df.loc[ib, "tariff"] = 25.0

    # --- Annex III: temporary ~15% (Clause 5, expires Dec 31, 2027) ---
    # Flat 15% approximation; true rate = max(0, 15% - Column1 MFN duty)
    iii = _mask_annex_III(df) & ~auto & ~ia & ~ib  # single application
    if cty == "4621":       # Russia: Clause (5)(c) → 25% (non-NTR)
        df.loc[iii & _is_alu_chapter(df), "tariff"] = 200.0  # Clause (8)
        df.loc[iii & ~_is_alu_chapter(df), "tariff"] = 25.0
    else:
        df.loc[iii, "tariff"] = 15.0

    # --- Annex II: removed from scope (Clause 4) ---
    ii = _mask_annex_II_removed(df) & ~auto
    df.loc[ii, "tariff"] = 0.0

    return df


def _apply_annex_III_expiry(df, cty, rate):
    """Jan 1, 2028: Annex III temporary reductions expire (Clause 7).
    Products revert to Annex I-B rates (Clause 3): 25% (UK 15%)."""
    auto = _mask_auto(df)
    ia = _mask_annex_IA(df)
    iii = _mask_annex_III(df) & ~auto & ~ia

    if cty == "4120":       # UK
        df.loc[iii, "tariff"] = 15.0
    elif cty == "4621":     # Russia
        df.loc[iii & _is_alu_chapter(df), "tariff"] = 200.0
        df.loc[iii & ~_is_alu_chapter(df), "tariff"] = 25.0
    else:
        df.loc[iii, "tariff"] = 25.0

    return df


In [ ]:
# ── Tariff Action Registry ────────────────────────────────────────────────
# Each action has a "phase" tag:
#   Phase 1: Base country-level rates (IEEPA layer). These may do blanket
#            df["tariff"] = rate resets.
#   Phase 2: Sector overrides (Section 232) and product exemptions.
#            These only touch specific HS codes and are never wiped by
#            phase-1 resets.
#
# The engine runs all phase-1 actions first, then all phase-2 actions.
# This matches the original code's structure where IEEPA base rates were
# set first and sector tariffs were applied afterward.

TARIFF_ACTIONS = [

    # ── Phase 1: IEEPA Base Country Rates ───────────────────────────────

    {
        "effective_date": "2025-02-04",
        "authority": "IEEPA",
        "reference": "EO 14193 (Canada), EO 14194 (Mexico), EO 14195 (China)",
        "description": "Fentanyl emergency — 20% on China/HK",
        "phase": 1,
        "apply": _apply_fentanyl_feb,
    },
    {
        "effective_date": "2025-03-06",
        "authority": "IEEPA",
        "reference": "Amendments to EO 14193/14194",
        "description": "USMCA exception; Canada 8.75%, Mexico 4.5%; China/HK 20%",
        "phase": 1,
        "apply": _apply_usmca_march,
    },
    {
        "effective_date": "2025-04-05",
        "authority": "IEEPA",
        "reference": "EO 14257",
        "description": "Reciprocal tariffs activate — 10% global baseline; country rates from Annex I",
        "phase": 1,
        "apply": _apply_reciprocal_apr5,
    },
    {
        "effective_date": "2025-04-08",
        "authority": "IEEPA",
        "reference": "Amendment to EO 14257",
        "description": "China/HK reciprocal to 84% (+20% fentanyl = 104%)",
        "phase": 1,
        "apply": _apply_china_escalation,
    },
    {
        "effective_date": "2025-04-09",
        "authority": "IEEPA",
        "reference": "Amendment to EO 14257 (Apr 9 announcement, immediate effect)",
        "description": "90-day pause: all countries to 10%; China/HK to 125%",
        "phase": 1,
        "apply": _apply_90day_pause,
    },
    {
        "effective_date": "2025-05-12",
        "authority": "IEEPA",
        "reference": "U.S.-China Joint Statement (Geneva, May 12, 2025)",
        "description": "China/HK reciprocal from 125% to 10% (+20% fentanyl = 30%)",
        "phase": 1,
        "apply": _apply_geneva_deal,
    },
    {
        "effective_date": "2025-08-07",
        "authority": "IEEPA",
        "reference": "EO 14326 (reciprocal), EO 14323 (Brazil), EO 14325 (Canada)",
        "description": "August country-specific rates; Brazil 40%; Canada fentanyl to 35%",
        "phase": 1,
        "apply": _apply_august_base_rates,
    },
    {
        "effective_date": "2025-11-01",
        "authority": "IEEPA",
        "reference": "EO 14358 (reciprocal extension), EO 14357 (fentanyl reduction)",
        "description": "China 1-year truce: 10% recip + 10% fentanyl = 20%; Canada scaled up",
        "phase": 1,
        "apply": _apply_november_china_base,
    },
    {
        "effective_date": "2025-11-14",
        "authority": "IEEPA",
        "reference": "U.S.-Switzerland Joint Statement",
        "description": "Switzerland rate capped at 15%",
        "phase": 1,
        "apply": _apply_november_swiss_base,
    },
    {
        "effective_date": "2025-12-04",
        "authority": "IEEPA",
        "reference": "ITA notice (90 FR 55964)",
        "description": "South Korea deal — 15% reciprocal ceiling",
        "phase": 1,
        "apply": _apply_korea_base,
    },
    {
        "effective_date": "2026-02-07",
        "authority": "IEEPA",
        "reference": "EO suspending EO 14329; U.S.-India interim framework",
        "description": "India: Russian oil tariff removed, reciprocal reduced to 18%",
        "phase": 1,
        "apply": _apply_india_deal,
    },
    {
        "effective_date": "2026-02-20",
        "authority": "Executive Order",
        "reference": "EO 'Ending Certain Tariff Actions' (Feb 20, 2026)",
        "description": "All IEEPA ad valorem duties revoked; 232/301 preserved; base rates zeroed",
        "phase": 1,
        "apply": _apply_scotus_base,
    },

    {
        "effective_date": "2026-02-24",
        "authority": "Section 122",
        "reference": "Proclamation 'Imposing a Temporary Import Surcharge' (Feb 20, 2026)",
        "description": "10% universal surcharge (150-day BOP emergency); Canada/Mexico USMCA exempt",
        "phase": 1,
        "apply": _apply_s122_base,
    },

    # ── Phase 2: Section 232 Sector Overrides & Exemptions ──────────────

    {
        "effective_date": "2025-03-12",
        "authority": "Section 232",
        "reference": "Proclamation 10895 (aluminum), Proclamation 10896 (steel)",
        "description": "Steel and aluminum tariffs at 25% globally",
        "phase": 2,
        "apply": _apply_steel_alu_25,
    },
    {
        "effective_date": "2025-04-03",
        "authority": "Section 232",
        "reference": "Proclamation 10908",
        "description": "25% tariff on automobiles and parts (Canada/Mexico USMCA exempt)",
        "phase": 2,
        "apply": _apply_auto_initial,
    },
    {
        "effective_date": "2025-04-11",
        "authority": "IEEPA",
        "reference": "Amendment to EO 14257 (Apr 11)",
        "description": "Consumer electronics exempt from reciprocal tariffs",
        "end_date": "2026-02-19",
        "phase": 2,
        "apply": _apply_phone_exemption,
    },
    {
        "effective_date": "2025-04-11",
        "authority": "IEEPA",
        "reference": "EO 14257 Annex II",
        "description": "Annex II product exemptions from reciprocal tariffs",
        "end_date": "2026-02-19",
        "phase": 2,
        "apply": _apply_annex_ii,
    },
    {
        "effective_date": "2025-06-03",
        "authority": "Section 232",
        "reference": "Proclamation 10947",
        "description": "Steel and aluminum tariffs raised to 50% (UK stays at 25%)",
        "phase": 2,
        "apply": _apply_steel_alu_50,
    },
    {
        "effective_date": "2025-08-07",
        "authority": "Section 232 / IEEPA",
        "reference": "Proclamation 10962 (copper), Proc 10908 amendments (autos), EO 14323 Annex (Brazil)",
        "description": "Copper 50%; auto deals (JP/EU/KR 15%); Brazil exemptions",
        "end_date": "2026-02-19",
        "phase": 2,
        "apply": _apply_august_sectors,
    },
    {
        "effective_date": "2025-11-14",
        "authority": "IEEPA",
        "reference": "EO 14360 (ag exemptions)",
        "description": "Agricultural product exemptions; China phone fentanyl to 10%",
        "end_date": "2026-02-19",
        "phase": 2,
        "apply": _apply_november_ag_phones,
    },
    {
        "effective_date": "2025-12-04",
        "authority": "Section 232",
        "reference": "ITA notice (90 FR 55964)",
        "description": "South Korea auto deal at 15%",
        "phase": 2,
        "apply": _apply_korea_auto,
    },
    {
        "effective_date": "2026-02-20",
        "authority": "Section 232 (surviving)",
        "reference": "EO 'Ending Certain Tariff Actions' — 232 preserved",
        "description": "Re-apply only surviving 232 tariffs: steel, alu, autos, copper",
        "phase": 2,
        "apply": _apply_scotus_sectors,
    },
    {
        "effective_date": "2026-02-24",
        "authority": "Section 122",
        "reference": "Proclamation Annex I, subdivision (aa)(ii)",
        "description": "Product exemptions from surcharge (energy, minerals, pharma, ag, electronics, etc.)",
        "phase": 2,
        "apply": _apply_s122_exemptions,
    },

    {
        "effective_date": "2026-04-06",
        "authority": "Section 232",
        "reference": "Proclamation 'Strengthening Actions...' (Apr 2, 2026)",
        "description": "Metals tiered: I-A 50%, I-B 25%, III ~15% temp, II exempt; RU alu 200%",
        "phase": 2,
        "apply": _apply_metals_restructuring,
    },
    {
        "effective_date": "2028-01-01",
        "authority": "Section 232",
        "reference": "Clause (7) — Annex III products revert to Annex I-B rates",
        "description": "Annex III temporary reductions expire; products revert to 25% (UK 15%)",
        "phase": 2,
        "apply": _apply_annex_III_expiry,
    },

]

### Tariff assignment engine

In [ ]:
# ── Core Tariff Assignment Engine ─────────────────────────────────────────

def tariff_imports(df, cnty_code, tariff_rate, date):
    """
    Apply all tariff actions effective on or before `date` to the HS10-level
    import DataFrame `df` for country `cnty_code`.

    The engine makes two passes:
      Phase 1 — Base country-level rates (IEEPA layer). These may do blanket
                df["tariff"] = rate resets.
      Phase 2 — Sector overrides (Section 232) and product exemptions.
                These only touch specific HS codes and are never wiped.

    Actions with an "end_date" field are skipped once `date` exceeds it.
    This prevents revoked IEEPA exemptions from leaking into the
    post-revocation regime.

    This two-phase structure ensures that Section 232 tariffs always override
    the IEEPA base rate, matching the original code's semantics.

    Parameters
    ----------
    df : DataFrame with columns: CTY_CODE, HS10, HS8, HS6, share, imports, duty
    cnty_code : str — Census country code
    tariff_rate : float — Country reciprocal rate (rtariff or rtariff-aug)
    date : pd.Timestamp — Effective date to evaluate

    Returns
    -------
    df with "tariff" column set
    """
    df["tariff"] = 0.0

    for phase in [1, 2]:
        for action in TARIFF_ACTIONS:
            eff = date >= pd.to_datetime(action["effective_date"])
            end = "end_date" not in action or date <= pd.to_datetime(action["end_date"])
            if action["phase"] == phase and eff and end:
                df = action["apply"](df, cnty_code, tariff_rate)

    return df

### Configuration

In [ ]:
# Configuration: set the weight year here
WEIGHT_YEAR = 2024

date_strings = [
    "2025-02-04",   # Fentanyl emergency (IEEPA)
    "2025-03-06",   # USMCA carve-outs (IEEPA)
    "2025-03-12",   # Steel & aluminum 25% (Section 232)
    "2025-04-03",   # Autos 25% (Section 232)
    "2025-04-05",   # Reciprocal tariffs activate (IEEPA)
    "2025-04-08",   # China escalation to 104% (IEEPA)
    "2025-04-09",   # 90-day pause; China 125% (IEEPA)
    "2025-04-11",   # Electronics exemption (IEEPA)
    "2025-05-12",   # Geneva deal — China reduced (IEEPA)
    "2025-06-03",   # Steel/alu to 50% (Section 232)
    "2025-08-07",   # August package (IEEPA/232)
    "2025-11-01",   # Nov China deal (IEEPA)
    "2025-11-14",   # Swiss deal + ag exemptions (IEEPA)
    "2025-12-04",   # South Korea deal (IEEPA/232)
    "2026-02-07",   # India deal (IEEPA)
    "2026-02-20",   # IEEPA tariffs revoked by EO
    "2026-02-24",   # Section 122 surcharge takes effect (10% universal)
    "2026-04-06",   # Section 232 metals tiered restructuring
]

## Download 2025 Import DataOnly needs to run once; results are cached as parquet files.

In [ ]:
# ── Download 2025 import data for revenue estimation ───────────────────────
# Only needs to run once; results are cached as parquet files.
# Skips countries that already have a cached file.

import os

data_dir = "./data"
skipped = 0
downloaded = 0

for _, row in country_list.iterrows():
    cty = row['CTY_CODE']
    outfile = f"{data_dir}/2025-hs10-imports{cty}.parquet"
    if os.path.exists(outfile):
        skipped += 1
        continue
    print(f"Downloading {row.get('CTY_NAME', cty)}...")
    df = download_imports(cty, year=2025)
    pq.write_table(pa.Table.from_pandas(df), outfile)
    downloaded += 1

print(f"Done. Downloaded: {downloaded}, skipped (cached): {skipped}")

## Revenue Computation

In [ ]:
# ── 2025 Tariff Revenue Estimate (Static/Mechanical) ───────────────────────
# Estimates revenue from executive-action tariffs using 2025 import values.
# For each interval between policy dates, applies the tariff schedule to
# HS10-level imports and prorates by number of days in the interval.
#
# Uses actual 2025 annual import values (from Census year-to-date data),
# spread evenly across the year. Tariff rates come from the policy engine.
# Caveat: cannot observe within-year timing of trade flows (e.g. front-loading
# in Q1, collapse during 125% China tariffs). Revenue is prorated by days
# in each interval assuming uniform monthly imports.

import warnings
warnings.filterwarnings('ignore')

# ── Realization adjustment ─────────────────────────────────────────────────
# Statutory tariff rates overstate actual collected duties due to exemptions,
# exclusions, FTZ entries, drawback, misclassification, de minimis, etc.
# This factor scales statutory rates to approximate realized collection rates.
# Default: 0.7 (realized ~10% vs. statutory ~14% in aggregate).
# Can be calibrated: set REALIZATION_FACTOR = actual_collected / statutory_estimate.
REALIZATION_FACTOR = 0.523

# ── Actual 2025 duty collected from Census data ───────────────────────────
actual_duty_rows = []
for _, row in country_list.iterrows():
    cty = row['CTY_CODE']
    df = load_imports(cty, year=2025)
    is_232 = _mask_steel(df) | _mask_alu(df) | _mask_auto(df) | _mask_copper(df)
    actual_duty_rows.append({
        'country': df['CTY_NAME'].iloc[0] if 'CTY_NAME' in df.columns else cty,
        'cty_code': cty,
        'total_imports': df['imports'].sum(),
        'total_duty': df['duty'].sum(),
        'duty_232': df.loc[is_232, 'duty'].sum(),
        'duty_non232': df.loc[~is_232, 'duty'].sum(),
    })
actual_df = pd.DataFrame(actual_duty_rows)
actual_total_duty = actual_df['total_duty'].sum()
actual_total_imports = actual_df['total_imports'].sum()

# Also compute 2024 baseline duty (what would have been collected without
# executive actions) to isolate incremental revenue
baseline_duty_2024 = 0.0
for _, row in country_list.iterrows():
    df24 = load_imports(row['CTY_CODE'], year=WEIGHT_YEAR)
    baseline_duty_2024 += df24['duty'].sum()

print(f"Actual 2025 customs duty collected (Census):  ${actual_total_duty/1e9:.1f}B")
print(f"  on total imports of:                        ${actual_total_imports/1e9:.1f}B")
print(f"  effective realized rate:                     {100*actual_total_duty/actual_total_imports:.1f}%")
print(f"Baseline 2024 duty (no exec actions):          ${baseline_duty_2024/1e9:.1f}B")
print(f"Implied incremental revenue (2025 - baseline): ${(actual_total_duty - baseline_duty_2024)/1e9:.1f}B")
print()

# ── Build intervals clipped to 2025 ────────────────────────────────────────
start_2025 = pd.to_datetime("2025-01-01")
end_2025   = pd.to_datetime("2025-12-31")

policy_dates = sorted([pd.to_datetime(d) for d in date_strings])
boundaries = [start_2025] + [d for d in policy_dates if start_2025 < d <= end_2025] + [end_2025]
boundaries = sorted(set(boundaries))

intervals = []
for i in range(len(boundaries) - 1):
    int_start = boundaries[i]
    int_end   = boundaries[i + 1]
    regime_date = int_start if int_start in policy_dates else max(
        [d for d in policy_dates if d <= int_start], default=start_2025
    )
    days = (int_end - int_start).days
    if days > 0:
        intervals.append((regime_date, int_start, int_end, days))

print(f"Revenue estimation: {len(intervals)} intervals covering "
      f"{sum(d for _,_,_,d in intervals)} days in 2025\n")

# ── Compute revenue by interval × country ──────────────────────────────────
rev_rows = []

for regime_date, int_start, int_end, days in intervals:

    for _, row in country_list.iterrows():
        cty = row['CTY_CODE']
        df = load_imports(cty, year=2025)

        rate = row["rtariff-aug"] if regime_date >= pd.to_datetime("2025-08-07") else row["rtariff"]
        result = tariff_imports(df, cty, rate, regime_date)

        is_232 = _mask_steel(result) | _mask_alu(result) | _mask_auto(result) | _mask_copper(result)

        # Statutory revenue = (tariff/100) × annual_imports × (days/365)
        result['rev_statutory'] = (result['tariff'] / 100.0) * result['imports'] * (days / 365)

        rev_232  = result.loc[is_232,  'rev_statutory'].sum()
        rev_ieepa = result.loc[~is_232, 'rev_statutory'].sum()

        rev_rows.append({
            'interval_start': int_start,
            'interval_end': int_end,
            'days': days,
            'country': result['CTY_NAME'].iloc[0] if 'CTY_NAME' in result.columns else cty,
            'cty_code': cty,
            'rev_ieepa': rev_ieepa,
            'rev_232': rev_232,
            'rev_total': rev_ieepa + rev_232,
        })

rev_df = pd.DataFrame(rev_rows)

# ── Summary ────────────────────────────────────────────────────────────────
total_ieepa = rev_df['rev_ieepa'].sum()
total_232   = rev_df['rev_232'].sum()
total_all   = rev_df['rev_total'].sum()

print("2025 Tariff Revenue Estimate (2025 imports, uniform monthly spread)")
print("=" * 75)
print(f"{'':>35s} {'Statutory':>12s}  {'Realized':>12s}")
print(f"{'':>35s} {'':>12s}  {'(×' + str(REALIZATION_FACTOR) + ')':>12s}")
print(f"{'-'*75}")
print(f"  {'IEEPA tariffs:':<33s} ${total_ieepa/1e9:>10.1f}B  ${total_ieepa*REALIZATION_FACTOR/1e9:>10.1f}B")
print(f"  {'Section 232 tariffs:':<33s} ${total_232/1e9:>10.1f}B  ${total_232*REALIZATION_FACTOR/1e9:>10.1f}B")
print(f"  {'Total (exec actions):':<33s} ${total_all/1e9:>10.1f}B  ${total_all*REALIZATION_FACTOR/1e9:>10.1f}B")
print()
print(f"  Actual 2025 duty collected (Census):        ${actual_total_duty/1e9:>10.1f}B")
print(f"  Less 2024 baseline duty:                   -${baseline_duty_2024/1e9:>10.1f}B")
print(f"  Implied incremental (actual):               ${(actual_total_duty - baseline_duty_2024)/1e9:>10.1f}B")
print()
implied_factor = (actual_total_duty - baseline_duty_2024) / total_all if total_all > 0 else np.nan
print(f"  Implied realization factor:  {implied_factor:.3f}")
print(f"  Current setting:             {REALIZATION_FACTOR:.3f}")
print()

# ── Revenue by interval ────────────────────────────────────────────────────
interval_rev = (rev_df.groupby(['interval_start', 'interval_end', 'days'])
                .agg({'rev_ieepa': 'sum', 'rev_232': 'sum', 'rev_total': 'sum'})
                .reset_index()
                .sort_values('interval_start'))

print("Revenue by policy interval (statutory):")
print("-" * 85)
print(f"{'Start':>12s}  {'End':>12s}  {'Days':>4s}  {'IEEPA':>12s}  {'232':>12s}  {'Total':>12s}  {'Ann. Rate':>10s}")
print("-" * 85)
for _, r in interval_rev.iterrows():
    ann_rate = r['rev_total'] / r['days'] * 365
    print(f"{str(r['interval_start'].date()):>12s}  {str(r['interval_end'].date()):>12s}  "
          f"{r['days']:>4.0f}  ${r['rev_ieepa']/1e9:>10.1f}B  ${r['rev_232']/1e9:>10.1f}B  "
          f"${r['rev_total']/1e9:>10.1f}B  ${ann_rate/1e9:>8.1f}B/yr")

# ── Top countries by IEEPA revenue ─────────────────────────────────────────
print()
country_rev = (rev_df.groupby('country')
               .agg({'rev_ieepa': 'sum', 'rev_232': 'sum', 'rev_total': 'sum'})
               .sort_values('rev_ieepa', ascending=False))
country_rev['pct'] = 100 * country_rev['rev_ieepa'] / total_ieepa

print("Top 10 countries by IEEPA tariff revenue (statutory):")
print("-" * 65)
for name, r in country_rev.head(10).iterrows():
    print(f"  {name:<25s}  ${r['rev_ieepa']/1e9:>7.1f}B  ({r['pct']:>5.1f}%)")